# Step 3 gate: every agent tool, called by hand on m-eurosat
Runs in a **sandbox workspace** (`GOTRAIN_WORKSPACE=.sandbox/workspace`): its own ledger, runs, logs and vault;
`finalize` and the engine's `final_test` refuse there, so the test split is unreachable.
Every step asserts the outcome it expects and stops the notebook otherwise; run ids come from the launch results.

In [ ]:
import os, re, shutil, sys, time
from pathlib import Path

REPO = Path.cwd().resolve().parent
SANDBOX = REPO / ".sandbox"
shutil.rmtree(SANDBOX, ignore_errors=True)               # fresh sandbox every time
os.environ["GOTRAIN_WORKSPACE"] = str(SANDBOX / "workspace")  # must be set before importing gotrain
sys.path.insert(0, str(REPO))

import gotrain
from gotrain import runs
from gotrain.tools.train import list_backbones, get_budget, launch_training, check_run, kill_run
from gotrain.tools.diagnose import evaluate_run, compare_runs, query_ledger
from gotrain.tools.finalize import finalize
from IPython.display import Image, display
%matplotlib inline

assert gotrain.is_sandbox(), "not in a sandbox: stop"
REAL_LEDGER = REPO / "workspace" / "ledger.jsonl"
real_ledger_lines = REAL_LEDGER.read_text().count("\n") if REAL_LEDGER.exists() else 0
TASK = "m-eurosat"

def show(result):
    """Print a tool result the way the agent receives it (text + images); return it."""
    print(f"[{result['status']}]")
    for c in result["content"]:
        if "text" in c:
            print(c["text"])
        else:
            display(Image(data=c["image"]["source"]["bytes"], width=700))
    return result

def launched(result):
    """The run id from a successful launch_training result; stop if the launch failed."""
    show(result)
    assert result["status"] == "success", "launch failed: stop"
    return re.search(r"Launched (\S+)", result["content"][0]["text"]).group(1)

def expect_error(result, contains):
    show(result)
    assert result["status"] == "error" and contains in result["content"][0]["text"], f"expected an error about {contains!r}"

def wait(run_id, min_epoch=None):
    while True:
        c = runs.check(run_id)
        if c["status"] != "running" or (min_epoch and c["epoch"] >= min_epoch):
            return c
        time.sleep(2)

## Plan: model zoo + budget

In [ ]:
show(list_backbones(TASK));

In [ ]:
show(get_budget(TASK));

## Launch: a bad config is rejected with the reason; a good one starts in the background

In [ ]:
expect_error(launch_training(TASK, {"backbone": "resnet50", "weights": "ssl4eo_moco", "lr": 5}, "bad lr on purpose"),
             "out of range")

In [ ]:
RUN_A = launched(launch_training(TASK, {"backbone": "resnet18", "weights": "imagenet", "epochs": 3},
                                 "Tool check: short ResNet18 ImageNet run."))

Only one training at a time:

In [ ]:
expect_error(launch_training(TASK, {"epochs": 3}, "should be refused while RUN_A runs"), "one training at a time")

In [ ]:
show(check_run(RUN_A))   # while running
assert wait(RUN_A)["status"] == "done"
show(check_run(RUN_A));

## Evaluate: metrics + confusion matrix + worst mistakes (as the agent sees them)

In [ ]:
assert show(evaluate_run(RUN_A))['status'] == 'success'

In [ ]:
assert show(evaluate_run(RUN_A, tta=True))['status'] == 'success'

## A child run (lineage + delta vs parent), then compare

In [ ]:
RUN_B = launched(launch_training(TASK, {"backbone": "resnet18", "weights": "imagenet", "epochs": 3, "lr": 0.0003},
                                 "Tool check: lower lr than the parent.", parent_run_id=RUN_A))
assert wait(RUN_B)["status"] == "done"
c = runs.check(RUN_B); assert c["delta_vs_parent"] is not None
show(check_run(RUN_B));

In [ ]:
assert show(compare_runs([RUN_A, RUN_B]))['status'] == 'success'

## Kill: stop a long run after 2 epochs; its best checkpoint is kept

In [ ]:
RUN_C = launched(launch_training(TASK, {"backbone": "resnet18", "weights": "imagenet", "epochs": 30},
                                 "Tool check: long run, to be killed after 2 epochs."))
wait(RUN_C, min_epoch=2)
show(kill_run(RUN_C, "tool check: kill path"))
assert runs.check(RUN_C)["status"] == "killed"
show(check_run(RUN_C));

## Failure: a run that runs out of GPU memory is reported with a hint

In [ ]:
RUN_D = launched(launch_training(TASK, {"backbone": "resnet50", "weights": "imagenet", "img_size": 512,
                                        "batch_size": 512, "epochs": 1, "warmup_epochs": 0},
                                 "Tool check: deliberately too large for a T4, should fail with an OOM hint."))
c = wait(RUN_D)
show(check_run(RUN_D))
assert c["status"] == "failed" and "out of memory" in (c["error"] or "").lower()

## Ledger + budget after the runs

In [ ]:
show(query_ledger(TASK, sort='recent'));

In [ ]:
show(get_budget(TASK));

## Finalize: refused (sandbox), so no test data is touched

In [ ]:
expect_error(finalize("m-brick-kiln", RUN_A, False), "sandbox")
expect_error(finalize(TASK, RUN_C, False), "sandbox")

## The real workspace was not touched

In [ ]:
assert not (REPO / "workspace" / "vault.jsonl").exists()
assert (REAL_LEDGER.read_text().count("\n") if REAL_LEDGER.exists() else 0) == real_ledger_lines
print("real ledger unchanged, real vault absent; sandbox ledger:", runs.ledger.LEDGER)